In [58]:
import sys
import os

# Get the current working directory
cwd = os.getcwd()
print(f"Current Working Directory: {cwd}")

# Define the path to the 'mapelite' folder
# We assume the notebook is running from the root 'Quality-Diversity-...' folder
mapelite_path = os.path.join(cwd, 'mapelite')

# Add it to the system path so Python can find config.py, utils.py, etc.
if mapelite_path not in sys.path:
    sys.path.append(mapelite_path)
    print(f"Added '{mapelite_path}' to sys.path")

Current Working Directory: d:\dev\Quality-Diversity-for-Racing-Track-Design\mapelite


In [ ]:
import numpy as np
import glob
import pickle

from ribs.archives import ProximityArchive, AddStatus
from ribs.schedulers import Scheduler

from dask.distributed import Client, LocalCluster, as_completed

from utils import array_to_solution
from mapelite.evaluator import EvaluatorNoveltySearch
from emitter import CustomEmitter

In [60]:
from mapelite.config import (
    SOLUTION_DIM,
    BATCH_SIZE,
    CHECKPOINT_DIR,
    HEATMAP_DIR,
    CHECKPOINT_EVERY,
    INVALID_SCORE,
    ITERATIONS
)

EMBEDDING_DIM = 32
DEFAULT_THRESHOLD = 5
SEED = 67

In [61]:
# --- Initialize directories ---
# Create directories if they don't exist
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(HEATMAP_DIR, exist_ok=True)

In [62]:
# --- DASK SETUP ---
print("Setting up Dask LocalCluster...")
cluster = LocalCluster(processes=True, n_workers=BATCH_SIZE, threads_per_worker=1)
client = Client(cluster)
print(f"Dask Dashboard link: {client.dashboard_link}")

Setting up Dask LocalCluster...


d:\dev\Quality-Diversity-for-Racing-Track-Design\venv\Lib\site-packages\distributed\node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 51671 instead
  warnings.warn(
C:\Users\milob\AppData\Local\Programs\Python\Python312\Lib\contextlib.py:144: UserWarning: Creating scratch directories is taking a surprisingly long time. (3.57s) This is often due to running workers on a network file system. Consider specifying a local-directory to point workers to write scratch data to a local disk.
  next(self.gen)


Dask Dashboard link: http://127.0.0.1:51671/status


In [63]:
archive = ProximityArchive(
    solution_dim=SOLUTION_DIM,
    measure_dim=EMBEDDING_DIM,
    k_neighbors=15,
    novelty_threshold=DEFAULT_THRESHOLD,
    seed=SEED,
    local_competition=True
)

emitter = CustomEmitter(
    archive,
    solution_dim=SOLUTION_DIM,
    batch_size=BATCH_SIZE,
    # Last element (ID) can be arbitrarily large (ID=iteration + fractional_part)
    bounds=[(0, 600)] * (SOLUTION_DIM - 1) + [(0, float("inf"))]
)

scheduler = Scheduler(archive, [emitter])

evaluator = EvaluatorNoveltySearch()

# Scatter the evaluator to all workers once (avoids re-serializing the model every submit)
evaluator_future = client.scatter(evaluator, broadcast=True)

Loading model from embeddings/models/model_metrics_VAE/model_metrics_VAE_latent32.pth...
Model loaded with latent_dim=32


In [64]:
# # --- Pilot: measure pairwise distances in embedding space ---
# from scipy.spatial.distance import pdist

# PILOT_ITERS = 5  # number of batches to sample
# pilot_measures = []

# for _ in range(PILOT_ITERS):
#     sols = scheduler.ask()
#     sol_dicts = [array_to_solution(sol) for sol in sols]

#     futs = [client.submit(_eval_on_worker, evaluator_future, sol) for sol in sol_dicts]
#     gathered = [f.result() for f in as_completed(futs)]

#     for sol_id, ok, msg, score, desc in gathered:
#         if ok:
#             pilot_measures.append(desc)

#     # Still need to tell the scheduler so it doesn't break
#     objs, descs = zip(*[(score if ok else INVALID_SCORE, desc)
#                         for sol_id, ok, msg, score, desc in gathered])
#     scheduler.tell(list(objs), list(descs))

# pilot_measures = np.array(pilot_measures)
# dists = pdist(pilot_measures)

# print(f"Pilot: {len(pilot_measures)} valid embeddings collected")
# print(f"Pairwise distances — min={dists.min():.4f}, "
#       f"25th={np.percentile(dists, 25):.4f}, "
#       f"median={np.median(dists):.4f}, "
#       f"75th={np.percentile(dists, 75):.4f}, "
#       f"max={dists.max():.4f}, "
#       f"mean={dists.mean():.4f}")
# print(f"\nSuggested novelty_threshold range: {np.percentile(dists, 10):.4f} – {np.median(dists) * 0.5:.4f}")

In [65]:
# --------------------------------------------------------------
# Resume from latest pickle checkpoint if one exists
# --------------------------------------------------------------
checkpoints = sorted(glob.glob(f"{CHECKPOINT_DIR}checkpoint_*.pkl"))
start_iter = 0
global_best_score = INVALID_SCORE
global_best_id = None

if checkpoints:
    latest_ckpt = checkpoints[-1]
    with open(latest_ckpt, "rb") as f:
        state = pickle.load(f)
    scheduler           = state["scheduler"]
    # Get the latest archive instance from the scheduler
    archive             = scheduler.archive
    start_iter          = state["iteration"]
    global_best_score   = state["global_best_score"]
    global_best_id      = state["global_best_id"]
    print(f"[Resume] Loaded {latest_ckpt}, resuming from iteration {start_iter+1}")
else:
    print("[Resume] No checkpoint found — starting fresh.")

[Resume] No checkpoint found — starting fresh.


In [ ]:
def _eval_on_worker(evaluator, sol):
    """Thin wrapper so Dask receives the evaluator as an already-scattered future."""
    return evaluator.evaluate(sol)

def run_novelty_search(total_iters, start_iter=0):
    global global_best_score, global_best_id
    
    for i in range(start_iter, total_iters):
        
        # Ask the scheduler for a batch of solutions to evaluate
        sols = scheduler.ask()
        sol_dicts = [array_to_solution(sol) for sol in sols]
        
        # submit evaluation tasks to Dask and wait for results
        futs = [client.submit(_eval_on_worker, evaluator_future, sol) for sol in sol_dicts]
        gathered = [f.result() for f in as_completed(futs)]
        
        obj_list, clean_solutions = [], []
        for sol_id, ok, msg, score, desc in gathered:
            
            if not ok:
                print(f"Warning: clamping bad score for ID={sol_id} ({msg})")
                score = INVALID_SCORE
            else:
                print(f"Solution ID={sol_id} evaluated with score={score:.2f}")
                if score > global_best_score:
                    global_best_score = score
                    global_best_id = sol_id
            
            clean_solutions.append((score, desc))
            obj_list.append(score)
        
        # Tell the scheduler the results
        obj_batch, meas_batch = zip(*clean_solutions)
        
        new_elites_count = 0
        sub_elites_count = 0
        original_add = archive.add
        
        def tracked_add(*args, **kwargs):
            nonlocal new_elites_count, sub_elites_count
            res = original_add(*args, **kwargs)
            
            # Extract statuses (format varies slightly depending on pyribs version)
            statuses = None
            if isinstance(res, tuple):
                statuses = res[0]
            elif hasattr(res, 'status'):
                statuses = res.status
            elif isinstance(res, dict) and 'status' in res:
                statuses = res['status']
                
            if statuses is not None:
                arr = np.asarray(statuses)
                # Count insertions based on AddStatus mapping (NEW=2, IMPROVE_EXISTING=1)
                new_elites_count += int(np.sum(arr == AddStatus.NEW))
                sub_elites_count += int(np.sum(arr == AddStatus.IMPROVE_EXISTING))
                
            return res
        
        # Apply the temporary patch
        archive.add = tracked_add
        
        try:
            # Tell the scheduler the results (which uses our patched archive.add internally)
            scheduler.tell(list(obj_batch), list(meas_batch))
        finally:
            # Remove the patch immediately so the object remains picklable later
            if 'add' in archive.__dict__:
                del archive.__dict__['add']
        
        # Log statistics
        batch_best = max(obj_list) if obj_list else INVALID_SCORE
        print(f"Iteration {i+1} ended. Best in batch = {batch_best:.2f}")
        print(f"Global best so far: {global_best_score:.2f} (ID={global_best_id})")
        
        print(f"Archive Updates: {new_elites_count} new elites inserted, {sub_elites_count} elites substituted.")
        
        # print archive stats at each iteration
        data_archive = archive.data()
        if data_archive:
            arch_obj = data_archive["objective"]
            valid = arch_obj != INVALID_SCORE
            mean_val = np.mean(arch_obj[valid]) if np.any(valid) else 0.0
            best_val = np.max(arch_obj[valid]) if np.any(valid) else 0.0
            elites = archive.stats.num_elites
            print(f"Archive size={len(archive)}, elites={elites}, mean={mean_val:.2f}, best={best_val:.2f}")
        else:
            print("Archive still empty")
            
        # Checkpoint
        if (i + 1) % CHECKPOINT_EVERY == 0:
            ckpt_name = f"{CHECKPOINT_DIR}checkpoint_{i+1:04d}.pkl"
            with open(ckpt_name, "wb") as f:
                pickle.dump({
                    "scheduler": scheduler,
                    "iteration": i+1,
                    "global_best_score": global_best_score,
                    "global_best_id": global_best_id
                }, f)
            print(f"[Checkpoint] Saved {ckpt_name}")

In [67]:
# Run main loop
run_novelty_search(ITERATIONS, start_iter)

Emitter.ask() called for iteration 1
Solution ID=0.5307751158145231 evaluated with score=14.00
Solution ID=0.7299022650947331 evaluated with score=7.25
Solution ID=0.6727963784687216 evaluated with score=11.50
Solution ID=0.504081887006117 evaluated with score=8.25
Solution ID=0.8038642914305504 evaluated with score=12.50
Solution ID=0.44209522907317456 evaluated with score=13.50
Solution ID=0.9506707217888605 evaluated with score=11.75
Solution ID=0.06642615126818663 evaluated with score=13.25
Solution ID=0.8653562337791582 evaluated with score=6.75
Solution ID=0.4776041510385908 evaluated with score=22.25


TypeError: 'NoneType' object is not subscriptable